# Session 13: Practice - OOP with APIs

## Introduction

In Session 9 you learned to design classes, and in Session 7 you learned to call APIs. In this practice session, you'll combine both: you'll design a class that wraps an external API, so that calling the API, parsing the response, and reusing the result all live in one reusable, testable place instead of being copy-pasted every time you need the data.

We'll build a `CityWeather` class around the **OpenWeatherMap Current Weather API**, which requires a free API key — the perfect excuse to also practice storing secrets properly with a `.env` file, instead of hardcoding them in the notebook.

## Learning Objectives

By the end of this session, you will be able to:

1. Design a class around an external API: decide what it should store and what it should do
2. Call an API from inside a class method, using a securely stored API key
3. Store selected fields from a JSON response as instance attributes
4. Write a helper method that parses a raw API response into clean attributes
5. Handle invalid requests and missing data without crashing the whole program
6. Implement `describe()`, `to_dict()`, and a `to_dataframe()` helper for a collection of objects
7. Reuse the same class across many API calls, and compare the resulting objects

## Table of Contents

1. [Setup: API Key and `.env` File](#1.-Setup:-API-Key-and-.env-File)
2. [Designing a Class Around an API](#2.-Designing-a-Class-Around-an-API)
3. [Building the `CityWeather` Class](#3.-Building-the-CityWeather-Class)
4. [`describe()`, `to_dict()`, and `to_dataframe()`](#4.-describe(),-to_dict(),-and-to_dataframe())
5. [Reusing the Class Across Multiple Queries](#5.-Reusing-the-Class-Across-Multiple-Queries)
6. [Comparing Multiple Objects](#6.-Comparing-Multiple-Objects)
7. [Exercises](#7.-Exercises)

---

## 1. Setup: API Key and `.env` File

### Step 1: Create a free account

1. Go to https://home.openweathermap.org/users/sign_up and sign up (it's free)
2. Confirm your email if you're asked to

### Step 2: Get your API key

1. Log in, then go to https://home.openweathermap.org/api_keys
2. You'll see a default key already generated (usually called "Default") — or click "Generate" to create a new one
3. Copy the key: a long string like `a1b2c3d4e5f6...`

**Note:** a brand new key can take up to ~2 hours to activate. If you get a `401 Unauthorized` right after creating it, that's why — wait a bit and try again, it's not a mistake in your code.

### Step 3: Create your `variables.env` file

1. In this same folder (`s13_oop_apis_practice/`), create a new file named exactly `variables.env`
2. Add your key as a single line, with no quotes and no spaces around the `=`:

```
OPENWEATHER_API_KEY=a1b2c3d4e5f6...
```

3. Replace `a1b2c3d4e5f6...` with your actual key, then save the file
4. **Never** commit this file to version control — it should already be covered by `.gitignore`

### Why not just paste the key into the notebook?

API keys are secrets, not code. If you hardcode a key and later share the notebook, push it to GitHub, or email it to a classmate, the key goes with it. Storing it in a `.env`-style file that never gets committed means:

- The notebook works for anyone who has their **own** key, without edits
- You can't accidentally leak your key when sharing code
- The same pattern works for any secret: database passwords, other API tokens, etc.

Your folder should look like this:

```
s13_oop_apis_practice/
├── s13_oop_apis_practice.ipynb   <- this notebook
└── variables.env                 <- you create this (never committed)
```

The `python-dotenv` library reads `variables.env` and loads its contents into environment variables, which you then read with `os.getenv()`.

| Problem | Likely cause |
|---|---|
| `OPENWEATHER_API_KEY` is `None` | `variables.env` isn't in the same folder as the notebook, or the variable name is misspelled |
| `401 Unauthorized` right after creating the key | New keys can take up to ~2 hours to activate — wait and retry |
| `404 Not Found` | The city name doesn't match any OpenWeatherMap location — check spelling |

In [25]:
import os
import re

import requests
import pandas as pd
from dotenv import load_dotenv

load_dotenv("variables.env")
API_KEY = os.getenv("OPENWEATHER_API_KEY")

if API_KEY:
    print(f"API key loaded successfully! ({API_KEY[:4]}...)")
else:
    print("ERROR: API key not found. Check your variables.env file.")

BASE_URL = "https://api.openweathermap.org/data/2.5/weather"

API key loaded successfully! (091b...)


---

## 2. Designing a Class Around an API

Before writing any code, it helps to answer three questions about the object you're about to build:

| Question | Answer for `CityWeather` |
|---|---|
| **What does one object represent?** | The current weather for a single city, at the moment it was fetched |
| **What should it store (attributes)?** | City, country, temperature, "feels like" temperature, humidity, description, wind speed |
| **What should it be able to do (methods)?** | Fetch its own data from the API, parse the response, describe itself, export itself as a dict |

Notice the API call itself is just **one responsibility** among several. The class isn't "a function that calls an API" — it's an object that *happens to* know how to populate itself from an API, the same way a `Book` object might know how to load itself from a file.

---

## 3. Building the `CityWeather` Class

### 3.1 First, look at the raw response

Before designing attributes, let's see what the API actually returns. We'll call it once as a plain request, outside of any class, just to inspect the shape.

In [6]:
params = {"q": "Madrid", "appid": API_KEY, "units": "metric"}

response = requests.get(BASE_URL, params=params)

response.json()

{'coord': {'lon': -3.7026, 'lat': 40.4165},
 'weather': [{'id': 801,
   'main': 'Clouds',
   'description': 'few clouds',
   'icon': '02d'}],
 'base': 'stations',
 'main': {'temp': 17.21,
  'feels_like': 16.94,
  'temp_min': 14.59,
  'temp_max': 18.18,
  'pressure': 1028,
  'humidity': 75,
  'sea_level': 1028,
  'grnd_level': 948},
 'visibility': 10000,
 'wind': {'speed': 2.778, 'deg': 303, 'gust': 8.889},
 'clouds': {'all': 17},
 'dt': 1791962314,
 'sys': {'type': 5,
  'id': 50006275,
  'country': 'ES',
  'sunrise': 1791959049,
  'sunset': 1791999433},
 'timezone': 7200,
 'id': 3117735,
 'name': 'Madrid',
 'cod': 200}

The fields we care about are nested a few levels deep: `main.temp`, `main.feels_like`, `main.humidity`, `weather[0].description`, `wind.speed`, and `sys.country`. This is exactly the kind of structure a parsing helper method should hide from the rest of our code.

### 3.2 Fetching data from inside the class

The API call becomes `_fetch_data()`, and the parsing logic becomes `_parse_data()` — both **methods**, not attributes, but they follow the same naming convention from Session 11:

| Prefix | Convention | Meaning |
|--------|------------|---------|
| None | `method()` | Public — part of the class's intended interface, safe to call from outside |
| Single `_` | `_method()` | Protected — internal detail, not name-mangled, still technically callable from outside |
| Double `__` | `__method()` | Private — name-mangled to `_ClassName__method`, intended to never be called from outside |

Let's see all three called from *outside* the class, with a small throwaway example (not `CityWeather`) so we can try all three access levels side by side.

In [7]:
class AccessDemo:
    def public_method(self):
        return "called from outside: works, as expected"

    def _protected_method(self):
        return "called from outside: still works, but signals 'please don't'"

    def __private_method(self):
        return "called from outside: blocked by name mangling"


demo = AccessDemo()

demo.public_method()

'called from outside: works, as expected'

In [8]:
# Single underscore: Python lets you, it's just a social contract
demo._protected_method()

"called from outside: still works, but signals 'please don't'"

In [9]:
# Double underscore: this is the call a user of the class would naturally try...
try:
    demo.__private_method()
except AttributeError as e:
    print(f"Blocked: {e}")

# ...but the method still exists, just under its mangled name
demo._AccessDemo__private_method()

Blocked: 'AccessDemo' object has no attribute '__private_method'


'called from outside: blocked by name mangling'

Notice the difference: `_protected_method()` is called exactly like `public_method()` — Python doesn't stop you, it's purely a naming convention telling other developers "this is internal." `__private_method()` is actually rewritten by Python at class-definition time, so calling it by its original name from outside raises `AttributeError`; the only way in is through the mangled name, which defeats the purpose of calling it "private" in the first place.

Back in `CityWeather`, `_fetch_data()` and `_parse_data()` use a **single** underscore, not double, on purpose:

- They're implementation details a user of `CityWeather` shouldn't need to call directly — you just do `CityWeather("Madrid")` and the object populates itself
- But they're still worth calling directly while debugging (e.g. inspecting what `_fetch_data()` returned before parsing), so we don't want Python actively hiding them behind name mangling
- Single underscore is a plain **convention** — Python does nothing to enforce it, unlike `__method`, which Python actively renames

If we expected `CityWeather` to be subclassed and wanted to protect these methods from name clashes with a subclass's own methods, double underscore would be the right call — but that's not a concern here.

### 3.3 Handling invalid requests and missing data

An API call inside a class can fail for the same reasons any API call can fail: a typo'd city name (`404`), a bad or inactive key (`401`), or a network hiccup. Following the error-handling practices from Session 11, `_fetch_data` should raise a clear, specific exception instead of returning a broken object:

```python
def _fetch_data(self):
    params = {"q": self.city, "appid": API_KEY, "units": "metric"}
    response = requests.get(BASE_URL, params=params)

    if response.status_code == 404:
        raise ValueError(f"City '{self.city}' was not found by OpenWeatherMap")
    if response.status_code == 401:
        raise ValueError("Invalid or inactive API key — check your variables.env file")
    if response.status_code != 200:
        raise ValueError(f"OpenWeatherMap returned status {response.status_code} for '{self.city}'")

    return response.json()
```

Raising here — instead of inside `__init__` directly — means `CityWeather("Nowhereland")` fails immediately and clearly, rather than producing a half-built object with missing attributes that breaks confusingly later.

### 3.4 Parsing the response into attributes

`_parse_data` takes the raw JSON and assigns exactly the fields we decided on in Section 2 — nothing more.

### 3.5 Putting it all together

Here's the full class: `__init__` orchestrates the two private helpers, and the public attributes are only ever set in one place (`_parse_data`), so there's a single, clear source of truth for what a `CityWeather` object contains.

In [10]:
class CityWeather:
    def __init__(self, city):
        self.city = city
        data = self._fetch_data()
        self._parse_data(data)

    def _fetch_data(self):
        params = {"q": self.city, "appid": API_KEY, "units": "metric"}
        response = requests.get(BASE_URL, params=params)

        if response.status_code == 404:
            raise ValueError(f"City '{self.city}' was not found by OpenWeatherMap")
        if response.status_code == 401:
            raise ValueError("Invalid or inactive API key — check your variables.env file")
        if response.status_code != 200:
            raise ValueError(
                f"OpenWeatherMap returned status {response.status_code} for '{self.city}'"
            )

        return response.json()

    def _parse_data(self, data):
        self.country = data["sys"]["country"]
        self.temperature = data["main"]["temp"]
        self.feels_like = data["main"]["feels_like"]
        self.humidity = data["main"]["humidity"]
        self.description = data["weather"][0]["description"]
        self.wind_speed = data["wind"]["speed"]

Let's test it, including the error path.

In [11]:
madrid = CityWeather("Madrid")

madrid.temperature, madrid.description, madrid.humidity

(17.6, 'scattered clouds', 74)

In [12]:
#CityWeather("Thisisnotarealcityname")

ValueError: City 'Thisisnotarealcityname' was not found by OpenWeatherMap

---

## 4. `describe()`, `to_dict()`, and `to_dataframe()`

A class that wraps an API is far more useful once it can also **present itself**. We'll add two instance methods, and one standalone function for working with collections of instances:

| Method | Belongs to | Purpose |
|---|---|---|
| `describe()` | Instance | A short, human-readable summary — like `__str__` from Session 10 |
| `to_dict()` | Instance | A plain `dict` of the object's data — easy to serialize, combine, or inspect |
| `cities_to_dataframe(cities)` | Module-level function | Turns a *list* of objects into a `pandas.DataFrame`, by calling `to_dict()` on each |

We re-open the class with these two new methods added.

In [13]:
class CityWeather:
    def __init__(self, city):
        self.city = city
        data = self._fetch_data()
        self._parse_data(data)

    def _fetch_data(self):
        params = {"q": self.city, "appid": API_KEY, "units": "metric"}
        response = requests.get(BASE_URL, params=params)

        if response.status_code == 404:
            raise ValueError(f"City '{self.city}' was not found by OpenWeatherMap")
        if response.status_code == 401:
            raise ValueError("Invalid or inactive API key — check your variables.env file")
        if response.status_code != 200:
            raise ValueError(
                f"OpenWeatherMap returned status {response.status_code} for '{self.city}'"
            )

        return response.json()

    def _parse_data(self, data):
        self.country = data["sys"]["country"]
        self.temperature = data["main"]["temp"]
        self.feels_like = data["main"]["feels_like"]
        self.humidity = data["main"]["humidity"]
        self.description = data["weather"][0]["description"]
        self.wind_speed = data["wind"]["speed"]

    def describe(self):
        return (
            f"{self.city}, {self.country}: {self.temperature}°C "
            f"({self.description}), humidity {self.humidity}%"
        )

    def to_dict(self):
        return {
            "city": self.city,
            "country": self.country,
            "temperature": self.temperature,
            "feels_like": self.feels_like,
            "humidity": self.humidity,
            "description": self.description,
            "wind_speed": self.wind_speed,
        }


def cities_to_dataframe(cities):
    return pd.DataFrame([city.to_dict() for city in cities])

In [14]:
madrid = CityWeather("Madrid")

madrid.describe()

'Madrid, ES: 17.6°C (scattered clouds), humidity 74%'

In [15]:
madrid.to_dict()

{'city': 'Madrid',
 'country': 'ES',
 'temperature': 17.6,
 'feels_like': 17.35,
 'humidity': 74,
 'description': 'scattered clouds',
 'wind_speed': 2.778}

---

## 5. Reusing the Class Across Multiple Queries

This is the real payoff of wrapping the API in a class: fetching weather for ten cities takes the same one line of code, repeated, instead of ten copies of request-building and parsing logic.

Some city names might fail (typos, cities OpenWeatherMap doesn't recognize), so we wrap each call in `try` / `except`, the same pattern used for the PokeAPI batch in Session 11 — one bad city shouldn't stop the rest.

In [17]:
city_names = ["Madrid", "London", "Tokyo", "New York", "Beirut", "Hogsmeade"]

cities = []

for name in city_names:
    try:
        cities.append(CityWeather(name))
    except ValueError as e:
        print(f"Skipping '{name}': {e}")

len(cities)

Skipping 'Hogsmeade': City 'Hogsmeade' was not found by OpenWeatherMap


5

---

## 6. Comparing Multiple Objects

With a list of `CityWeather` objects, we can compare them directly — using `max()`/`min()` with a `key`, the same technique from the `Playlist.most_played()` exercise in Session 9 — or convert the whole collection into a DataFrame for a tabular view.

In [18]:
hottest = max(cities, key=lambda c: c.temperature)

hottest.describe()

'Tokyo, JP: 26.32°C (clear sky), humidity 61%'

In [19]:
weather_df = cities_to_dataframe(cities)

weather_df.sort_values("temperature", ascending=False)

,city,country,temperature,feels_like,humidity,description,wind_speed
2,Tokyo,JP,26.32,26.32,61,clear sky,2.060
4,Beirut,LB,25.77,26.13,66,overcast clouds,2.020
3,New York,US,19.85,20.08,84,clear sky,5.140
0,Madrid,ES,17.60,17.35,74,scattered clouds,2.778
1,London,GB,9.67,8.38,91,broken clouds,2.570


In [20]:
weather_df["temperature"].mean()

np.float64(19.842)

---

## 7. Exercises

### Exercise 1: `is_rainy()`

Add a method `is_rainy()` to `CityWeather` that returns `True` if `"rain"` appears anywhere in `self.description` (case-insensitive), `False` otherwise.

In [44]:
# Your code here: copy the CityWeather class from Section 4 and add is_rainy()
class CityWeather:
    def __init__(self, city):
        self.city = city
        data = self._fetch_data()
        self._parse_data(data)

    def _fetch_data(self):
        params = {"q": self.city, "appid": API_KEY, "units": "metric"}
        response = requests.get(BASE_URL, params=params)

        if response.status_code == 404:
            raise ValueError(f"City '{self.city}' was not found by OpenWeatherMap")
        if response.status_code == 401:
            raise ValueError("Invalid or inactive API key — check your variables.env file")
        if response.status_code != 200:
            raise ValueError(
                f"OpenWeatherMap returned status {response.status_code} for '{self.city}'"
            )

        return response.json()

    def _parse_data(self, data):
        self.country = data["sys"]["country"]
        self.temperature = data["main"]["temp"]
        self.feels_like = data["main"]["feels_like"]
        self.humidity = data["main"]["humidity"]
        self.description = data["weather"][0]["description"]
        self.wind_speed = data["wind"]["speed"]

    def describe(self):
        return (
            f"{self.city}, {self.country}: {self.temperature}°C "
            f"({self.description}), humidity {self.humidity}%"
        )

    def to_dict(self):
        return {
            "city": self.city,
            "country": self.country,
            "temperature": self.temperature,
            "feels_like": self.feels_like,
            "humidity": self.humidity,
            "description": self.description,
            "wind_speed": self.wind_speed,
        }

    def is_rainy(self):
        return "rain" in self.description

    def feels_colder(self):
        return self.feels_like < self.temperature


# Test your method

rainy_city = CityWeather("Madrid")

rainy_city.is_rainy()

False

### Exercise 2: `feels_colder()`

Add a method `feels_colder()` that returns `True` if `self.feels_like` is lower than `self.temperature` (i.e. wind or humidity make it feel colder than the thermometer says).

In [45]:
# Your code here: add feels_colder() to CityWeather


# Test your method
windy_city = CityWeather("Madrid")
windy_city.feels_colder()

True

### Exercise 3: `most_humid(cities)`

Write a standalone function `most_humid(cities)` that takes a list of `CityWeather` objects and returns the one with the highest `humidity`, using `max()` with a `key` argument (same pattern as `hottest` in Section 6).

In [52]:
def most_humid(cities):
    return max(cities, key= lambda t : float(t.humidity))


# Test your function
most_humid(cities).describe()

'London, GB: 9.67°C (broken clouds), humidity 91%'

### Exercise 4: `compare_cities(city_a, city_b)`

Write a function `compare_cities(city_a, city_b)` that takes two `CityWeather` objects and returns a sentence like:

```
"Madrid is 5.2°C hotter than London"
```

(or `"colder than"`, or `"the same temperature as"` if they're equal). Use `city_a.describe()`-style string formatting as a reference, but build this as a new, separate function.

In [19]:
def compare_cities(city_a, city_b):
    # Your code here
    pass


# Test your function
# compare_cities(cities[0], cities[1])

---

## Summary

In this session, you learned how to design a class around an external API:

1. **Secrets stay out of the notebook**: API keys live in `variables.env`, loaded with `python-dotenv` and read via `os.getenv()`
2. **Design before coding**: decide what one object represents, what it stores, and what it does, before writing `__init__`
3. **Fetching is one responsibility among several**: a private `_fetch_data()` method keeps the request logic separate from parsing and presentation
4. **Parsing is a dedicated helper**: `_parse_data()` turns a nested JSON response into flat, predictable attributes
5. **Fail clearly**: invalid cities or bad keys raise a specific `ValueError` with a useful message, instead of producing a broken object
6. **`describe()` and `to_dict()`** make a single object easy to read and easy to export
7. **Reuse and compare**: the same class powers one call or a hundred, and a module-level `to_dataframe()`-style function turns a list of objects into something you can sort, filter, and aggregate with pandas

### Key Takeaways

- Wrapping an API in a class pays off the moment you need to call it more than once
- Keep the API call, the parsing, and the presentation as separate methods — each one easy to test and reason about on its own
- The same `try`/`except` discipline from Session 11 applies here: one bad request in a batch shouldn't take down the rest